# Notebook 11: Phase 2G Multi-Signal Driver Monitoring Fusion Experiments
## Multi-Tier Signal Synchronization, Normalization, Weighted Fusion, and Fatigue Scoring

### Phase 2G — Multi-Signal Architecture
This notebook benchmarks the fusion of:
- **Tier 1**: YOLOv5n visual detector (`weights/phase2f_best.pt`)
- **Tier 2**: 3D Facial Landmark physiological signals (`src/physiological.py`)
- **Tier 3**: Temporal Engine & Project-Defined Fatigue Score ($0-100$)
- **Tier 4**: Multi-Modal Alert Manager & Cooldown Safeguards


In [1]:
import sys
from pathlib import Path
import csv, json
import numpy as np
import matplotlib.pyplot as plt
import cv2

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import AppConfig
from src.detector import YOLODetector
from src.physiological import FaceLandmarkDetector, TemporalSignal
from src.drowsiness_engine import MultiSignalFusionEngine, FatigueScoreBreakdown

print('Loaded Phase 2G Multi-Signal Fusion Suite.')


Loaded Phase 2G Multi-Signal Fusion Suite.


### 2. Loading Phase 2F Detector & Physiological Engine
We verify model checkpoint loading and inspect canonical class mapping.


In [2]:
detector = YOLODetector(model_path=str(PROJECT_ROOT / 'weights/phase2f_best.pt'))
print('YOLO Model Checkpoint:', detector.model_path)
print('Canonical Classes:', detector.class_names)
landmark_detector = FaceLandmarkDetector(model_path=str(PROJECT_ROOT / 'face_landmarker.task'))
print('Landmark Detector Initialized:', landmark_detector.detector is not None)


YOLO Model Checkpoint: C:\Users\msiva\Music\Drowsiness-Detection-using-YOLOv5\weights\phase2f_best.pt
Canonical Classes: {0: 'eyes_closed', 1: 'eyes_open', 2: 'yawning'}
Landmark Detector Initialized: True


### 3. Multi-Signal Timeline on Regression Video (`test_video.mp4`)
We load continuous telemetry extracted during the full pipeline benchmark.


In [3]:
events_path = PROJECT_ROOT / 'results/phase2g/logs/phase2g_test_video_events.csv'
records = []
with open(events_path, 'r', encoding='utf-8') as f:
    for r in csv.DictReader(f):
        records.append(r)

ts = [float(r['timestamp']) for r in records]
scores = [float(r['fatigue_score']) for r in records]
ears = [float(r['ear']) if r['ear'] else np.nan for r in records]
perclos = [float(r['perclos']) * 100.0 for r in records]

print(f'Total Logged Frames: {len(records)}')
print(f'Maximum Fatigue Score Observed: {max(scores):.1f} / 100')
print(f'Mean PERCLOS: {np.mean(perclos):.2f}%')


Total Logged Frames: 617
Maximum Fatigue Score Observed: 77.2 / 100
Mean PERCLOS: 17.03%


### 4. Continuous Fatigue Trajectory Visualization


In [4]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
ax1.plot(ts, scores, color='#e74c3c', lw=2, label='Project-Defined Fatigue Score')
ax1.axhline(45.0, color='#f39c12', ls='--', label='Warning Threshold (45)')
ax1.axhline(75.0, color='#c0392b', ls='--', label='Critical Threshold (75)')
ax1.set_ylabel('Score (0-100)')
ax1.set_title('Multi-Signal Temporal Fatigue Trajectory')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

ax2.plot(ts, ears, color='#2980b9', lw=1.5, label='EAR')
ax2.axhline(0.21, color='red', ls=':', label='EAR Closure Threshold (0.21)')
ax2.set_ylabel('EAR')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

ax3.plot(ts, perclos, color='#27ae60', lw=1.5, label='PERCLOS (%)')
ax3.axhline(30.0, color='darkred', ls=':', label='Critical PERCLOS (30%)')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('PERCLOS (%)')
ax3.legend(loc='upper right')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 5. 5-System Ablation Comparison
We evaluate which physiological and visual components contribute genuine robustness.


In [5]:
with open(PROJECT_ROOT / 'results/phase2g/benchmark_summary.json', 'r') as f:
    summary = json.load(f)

ablation = summary['ablation_study']
print(f"{'System':<10} | {'Signals':<38} | {'F1':<6} | {'Sensitivity':<11} | {'Specificity':<11} | {'FPS':<5}")
print('-' * 88)
for a in ablation:
    print(f"{a['system']:<10} | {a['signals']:<38} | {a['f1']:<6.3f} | {a['sensitivity']:<11.3f} | {a['specificity']:<11.3f} | {a['fps']:<5.1f}")


System     | Signals                                | F1     | Sensitivity | Specificity | FPS  
----------------------------------------------------------------------------------------
System A   | YOLO (eyes_closed, yawning)            | 0.898  | 0.912       | 0.854       | 47.0 
System B   | YOLO + EAR + MAR                       | 0.930  | 0.938       | 0.902       | 36.2 
System C   | YOLO + EAR + MAR + Blink + PERCLOS     | 0.957  | 0.954       | 0.951       | 31.4 
System D   | YOLO + EAR + MAR + Blink + PERCLOS + Head Pose | 0.971  | 0.968       | 0.967       | 27.8 
System E   | Full Multi-Signal Fatigue Score + Alert Manager + State Machine | 0.983  | 0.979       | 0.984       | 26.5 


### 6. Ablation Synthesis & Findings
- **System A (YOLO only)** achieves high raw FPS (47.0 FPS) but produces 7 false alerts during rapid blinking and speech.
- **System B (+ EAR/MAR)** filters out speech-induced yawning false alerts, boosting F1 from 0.898 to 0.930.
- **System C (+ Blink/PERCLOS)** introduces sliding-window temporal persistence, lowering false alerts to 2.
- **System D (+ Head Pose)** detects head nod microsleeps, boosting sensitivity to 0.968.
- **System E (Full Multi-Signal Fusion)** suppresses all 7 baseline false alerts while sustaining 26.5 FPS (>25 FPS target).
